# Spatial Beam Patterns

In this task, you must plot the spatial beam pattern given the channel measurements on the antenna elements. Let the coordinate of the antenna elements i be defined by (xi , yi ), and the
corresponding channel measurement at antenna element i be h_i. 

Therefore, the spatial gain (power) at some point (x, y) on the 2D plane can be given by:

<img src="media/image4.png" alt="drawing" width="600"/> 


where N is the number of antenna elements, and λ is the wavelength of the RF signal.

However, due to the reflective nature of RFID transmissions, the above formula will need to be modified. 

You should write down the modified formula in your report, and explain it. Further, you should complete the code based on the above modified formula, in the function
`spatial_beam_pattern` to return the Power value at some point (x, y) in 2D space. 

This function should take the following inputs:

- The coordinate (x,y).
- The matrix ant enna_pos, where the i t h row represents the tuple (xi , yi ).
- The channel vector h, where hi represents the channel observed at the i t h antenna element.
- λ which is the wavelength of the RF signal

In [ ]:
## TODO: Implement the following functions
import numpy as np
import matplotlib.pyplot as plt
import os


def spatial_beam_pattern(x, y, antenna_pos, h, wavelength):
    """
    Compute the spatial beam pattern at position (x, y) given antenna positions.
    Parameters:
        x : float
            x-coordinate of the point where the beam pattern is evaluated.
        y : float
            y-coordinate of the point where the beam pattern is evaluated.
        antenna_pos : np.ndarray
            Array of shape (N, 2) containing the (x, y) positions of N antennas.
        h : np.ndarray
            Array of shape (N,) containing the phase shifts for each antenna.
        wavelength : float
            Wavelength of the signal.
    Returns:
        power : float
            The computed power of the beam pattern at (x, y).
    """
    
    xi = antenna_pos[:, 0]
    yi = antenna_pos[:, 1]

    ri = np.sqrt((x - xi)**2 + (y - yi)**2)
    steering = np.exp(1j * 4 * np.pi / wavelength * ri)
    signal = np.exp(-1j * h) * steering

    power = np.abs(np.sum(signal))**2

    return power

def get_source_channel(antenna_coord, source, wavelength):
    xs = antenna_coord[:, 0]
    ys = antenna_coord[:, 1]
    sx, sy = source

    d = np.sqrt((sx - xs)**2 + (sy - ys)**2)
    channels = 4 * np.pi / wavelength * d

    return channels

def make_ant_pos(antenna_spacing, num_ant):
    xs = np.arange(num_ant) * antenna_spacing
    xs = xs - np.mean(xs)
    ys = np.zeros(num_ant)
    antenna_coord = np.column_stack((xs, ys))


    return antenna_coord

#### Testing on simulated data 
After completing the function, run the code cell below to test it on simulated data. The outputs should be saved to the `Results` folder and submitted along with your code

In [ ]:
# Parameters
lambda_ = 4
antenna_spacing = [lambda_/4, lambda_/2, lambda_]
num_ant = 4

source = [1000000, 1000000]
x_grid = np.arange(-1000, 1001)
y_grid = np.flip(np.arange(0, 2001))

# Loop through antenna spacings
for l in range(3):
    antenna_coord = make_ant_pos(antenna_spacing[l], num_ant)
    channels = get_source_channel(antenna_coord, source, lambda_)

    M = np.zeros((len(x_grid), len(y_grid)))

    for j in range(len(x_grid)):
        for k in range(len(y_grid)):
            M[j, k] = spatial_beam_pattern(x_grid[j], y_grid[k], antenna_coord, channels, lambda_)

    M = np.transpose(M)

    plt.figure()
    plt.imshow(M, extent=[x_grid.min(), x_grid.max(), y_grid.min(), y_grid.max()], cmap='gray', aspect='auto')
    plt.colorbar()
    plt.ylabel('Y Axis', fontsize=16)
    plt.xlabel('X Axis', fontsize=16)
    plt.title(f'Test Case Spatial Beam Pattern with Antenna Spacing {antenna_spacing[l]}', fontsize=20)



    plt.savefig(f'Results/Result_test_spatial_beam{antenna_spacing[l]}.png', format='png')
    plt.close()

In [ ]:
# import imageio
# img_debug = imageio.imread('Debugging_data/Result_test_spatial_beam1.0.png')
# img_result = imageio.imread('Results/Result_test_spatial_beam1.0.png')

# plt.figure(figsize=(15,5))

# plt.subplot(1,3,1)
# plt.title('Debug Image')
# plt.imshow(img_debug, cmap='gray')
# plt.axis('off')

# plt.subplot(1,3,2)
# plt.title('Result Image')
# plt.imshow(img_result, cmap='gray')
# plt.axis('off')

# plt.show()

#### Testing on Data from RFID Hardware

Below you will find the paramters of the channel values and positions of three antenna elements. 
- Plot the spatial beam pattern considering only antenna 1 and 2
- Plot the spatial beam pattern considering only antenna 1 and 3

In [ ]:
# Values of h1, h2 and h3 provided here
h1 = 66.2000  # In degrees
h2 = 109.6875  # In degrees
h3 = 164.4250  # In degrees

# Convert to radians
h1 = np.pi * h1 / 180
h2 = np.pi * h2 / 180
h3 = np.pi * h3 / 180

# Coordinates of Antenna Elements in 2D plane provided here
# Assume ant 1 is at origin and all other antennas lie on x axis
d = 7.62e-2
ant1 = np.array([0, 0])
ant2 = np.array([-d, 0])
ant3 = np.array([-2 * d, 0])

# Frequency and Lambda provided here
freq = 9.0275e8
wavelength = 3e8 / freq


In [ ]:
x_grid = np.linspace(-1, 1, 400)
y_grid = np.linspace(0, 2, 400)

def compute_pattern(antenna_pos, h, wavelength):
    M = np.zeros((len(x_grid), len(y_grid)))

    for j in range(len(x_grid)):
        for k in range(len(y_grid)):
            M[j, k] = spatial_beam_pattern(x_grid[j], y_grid[k], antenna_pos, h, wavelength)

    M = np.transpose(M)
    return M

# Antenna 1 and Antenna 2
antenna_pos_12 = np.vstack([ant1, ant2])
h_12 = np.array([h1, h2])
M12 = compute_pattern(antenna_pos_12, h_12, wavelength)

plt.figure(figsize=(7,6))
plt.imshow(M12, extent=[x_grid.min(), x_grid.max(), y_grid.min(), y_grid.max()], origin='lower', cmap='gray', aspect='auto')
plt.title('Spatial beam pattern - Antenna 1 & 2')
plt.xlabel('X Axis')
plt.ylabel('Y Axis')
plt.colorbar()
plt.show()

# Antenna 1 and Antenna 3
antenna_pos_13 = np.vstack([ant1, ant3])
h_13 = np.array([h1, h3])
M13 = compute_pattern(antenna_pos_13, h_13, wavelength)

plt.figure(figsize=(7,6))
plt.imshow(M13, extent=[x_grid.min(), x_grid.max(), y_grid.min(), y_grid.max()], origin='lower', cmap='gray', aspect='auto')
plt.title('Spatial beam pattern - Antenna 1 & 3')
plt.xlabel('X Axis')
plt.ylabel('Y Axis')
plt.colorbar()
plt.show()
